<a href="https://colab.research.google.com/github/Yisihaq01/breast-histology-calibration/blob/main/Notebooks/01_breakhis_data_audit_and_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Yisihaq01/breast-histology-calibration.git


Cloning into 'breast-histology-calibration'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 20 (delta 2), reused 10 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 7.08 KiB | 805.00 KiB/s, done.
Resolving deltas: 100% (2/2), done.


In [ ]:
%cd /content/breast-histology-calibration

/content/breast-histology-calibration


In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub

breakhis_path = kagglehub.dataset_download("ambarish/breakhis")

print("BreakHis dataset path:", breakhis_path)

100%|██████████| 3.99G/3.99G [00:50<00:00, 84.6MB/s]

Extracting files...


BreakHis dataset path: /root/.cache/kagglehub/datasets/ambarish/breakhis/versions/4


## 1. BreakHis Dataset Integrity Check

The BreakHis dataset is downloaded from the `ambarish/breakhis` Kaggle mirror.
Before analysis, we verify the total number of images, native image dimensions,
and magnification distribution against the expected BreakHis structure.

In [ ]:
from pathlib import Path
from PIL import Image
from collections import Counter
import re

root = Path(breakhis_path)

image_files = list(root.rglob("*.png"))

print("Dataset root:", root)
print("Total PNG images:", len(image_files))

# Image dimensions
size_counts = Counter()

for file in image_files:
    with Image.open(file) as img:
        size_counts[img.size] += 1

print("\nImage dimensions:")
for size, count in size_counts.most_common():
    print(f"{size}: {count}")

# Magnification counts
magnification_counts = Counter()

for file in image_files:
    match = re.search(r"-(40|100|200|400)-", file.name)

    if match:
        magnification_counts[match.group(1)] += 1

print("\nMagnification counts:")
for mag in ["40", "100", "200", "400"]:
    print(f"{mag}X: {magnification_counts[mag]}")

Dataset root: /root/.cache/kagglehub/datasets/ambarish/breakhis/versions/4
Total PNG images: 7909

Image dimensions:
(700, 460): 7835
(700, 456): 74

Magnification counts:
40X: 1995
100X: 2081
200X: 2013
400X: 1820


## 2. Metadata Extraction

BreakHis filenames contain information about diagnosis, histologic subtype,
patient/sample identity, magnification, and image number.

We extract these fields into a structured metadata table so that all subsequent
splitting is performed at the patient level rather than the image level.

In [ ]:
import pandas as pd

records = []

pattern = re.compile(
    r"^SOB_([BM])_([A-Z]+)-(.+)-(40|100|200|400)-(\d+)\.png$"
)

unparsed_files = []

for file in image_files:

    match = pattern.match(file.name)

    if not match:
        unparsed_files.append(file.name)
        continue

    diagnosis_code = match.group(1)
    subtype_code = match.group(2)
    patient_id = match.group(3)
    magnification = int(match.group(4))
    image_number = int(match.group(5))

    records.append({
        "filename": file.name,
        "filepath": str(file),
        "patient_id": patient_id,
        "diagnosis_code": diagnosis_code,
        "subtype_code": subtype_code,
        "magnification": magnification,
        "image_number": image_number
    })

metadata = pd.DataFrame(records)

print("Successfully parsed images:", len(metadata))
print("Unparsed images:", len(unparsed_files))

metadata.head()

Successfully parsed images: 7909
Unparsed images: 0


,filename,filepath,patient_id,diagnosis_code,subtype_code,magnification,image_number
0,SOB_B_A-14-22549AB-40-008.png,/root/.cache/kagglehub/datasets/ambarish/break...,14-22549AB,B,A,40,8
1,SOB_B_A-14-22549AB-40-007.png,/root/.cache/kagglehub/datasets/ambarish/break...,14-22549AB,B,A,40,7
2,SOB_B_A-14-22549AB-40-015.png,/root/.cache/kagglehub/datasets/ambarish/break...,14-22549AB,B,A,40,15
3,SOB_B_A-14-22549AB-40-018.png,/root/.cache/kagglehub/datasets/ambarish/break...,14-22549AB,B,A,40,18
4,SOB_B_A-14-22549AB-40-014.png,/root/.cache/kagglehub/datasets/ambarish/break...,14-22549AB,B,A,40,14


In [ ]:
subtype_names = {
    "A": "Adenosis",
    "F": "Fibroadenoma",
    "TA": "Tubular adenoma",
    "PT": "Phyllodes tumor",
    "DC": "Ductal carcinoma",
    "LC": "Lobular carcinoma",
    "MC": "Mucinous carcinoma",
    "PC": "Papillary carcinoma"
}

diagnosis_names = {
    "B": "Benign",
    "M": "Malignant"
}

metadata["diagnosis"] = metadata["diagnosis_code"].map(diagnosis_names)
metadata["subtype"] = metadata["subtype_code"].map(subtype_names)

metadata.head()

,filename,filepath,patient_id,diagnosis_code,subtype_code,magnification,image_number,diagnosis,subtype
0,SOB_B_A-14-22549AB-40-008.png,/root/.cache/kagglehub/datasets/ambarish/break...,14-22549AB,B,A,40,8,Benign,Adenosis
1,SOB_B_A-14-22549AB-40-007.png,/root/.cache/kagglehub/datasets/ambarish/break...,14-22549AB,B,A,40,7,Benign,Adenosis
2,SOB_B_A-14-22549AB-40-015.png,/root/.cache/kagglehub/datasets/ambarish/break...,14-22549AB,B,A,40,15,Benign,Adenosis
3,SOB_B_A-14-22549AB-40-018.png,/root/.cache/kagglehub/datasets/ambarish/break...,14-22549AB,B,A,40,18,Benign,Adenosis
4,SOB_B_A-14-22549AB-40-014.png,/root/.cache/kagglehub/datasets/ambarish/break...,14-22549AB,B,A,40,14,Benign,Adenosis


In [ ]:
print("=== FULL BREAKHIS DATASET ===")

print("\nTotal images:")
print(len(metadata))

print("\nUnique patients:")
print(metadata["patient_id"].nunique())

print("\nImages by diagnosis:")
print(metadata["diagnosis"].value_counts())

patient_metadata = (
    metadata[
        ["patient_id", "diagnosis", "subtype"]
    ]
    .drop_duplicates()
)

print("\nPatients by diagnosis:")
print(patient_metadata["diagnosis"].value_counts())

print("\nImages by subtype:")
print(metadata["subtype"].value_counts())

print("\nPatients by subtype:")
print(patient_metadata["subtype"].value_counts())

=== FULL BREAKHIS DATASET ===

Total images:
7909

Unique patients:
81

Images by diagnosis:
diagnosis
Malignant    5429
Benign       2480
Name: count, dtype: int64

Patients by diagnosis:
diagnosis
Malignant    58
Benign       24
Name: count, dtype: int64

Images by subtype:
subtype
Ductal carcinoma       3451
Fibroadenoma           1014
Mucinous carcinoma      792
Lobular carcinoma       626
Tubular adenoma         569
Papillary carcinoma     560
Phyllodes tumor         453
Adenosis                444
Name: count, dtype: int64

Patients by subtype:
subtype
Ductal carcinoma       38
Fibroadenoma           10
Mucinous carcinoma      9
Tubular adenoma         7
Papillary carcinoma     6
Lobular carcinoma       5
Adenosis                4
Phyllodes tumor         3
Name: count, dtype: int64


In [ ]:
breakhis_40x = (
    metadata[
        metadata["magnification"] == 40
    ]
    .copy()
    .reset_index(drop=True)
)

print("=== BREAKHIS 40X ===")

print("\nTotal 40X images:")
print(len(breakhis_40x))

print("\nUnique 40X patients:")
print(breakhis_40x["patient_id"].nunique())

print("\n40X images by diagnosis:")
print(breakhis_40x["diagnosis"].value_counts())

print("\n40X patients by diagnosis:")
print(
    breakhis_40x[
        ["patient_id", "diagnosis"]
    ]
    .drop_duplicates()
    ["diagnosis"]
    .value_counts()
)

print("\n40X images by subtype:")
print(
    breakhis_40x["subtype"]
    .value_counts()
)

print("\n40X patients by subtype:")
print(
    breakhis_40x[
        ["patient_id", "subtype"]
    ]
    .drop_duplicates()
    ["subtype"]
    .value_counts()
)

=== BREAKHIS 40X ===

Total 40X images:
1995

Unique 40X patients:
81

40X images by diagnosis:
diagnosis
Malignant    1370
Benign        625
Name: count, dtype: int64

40X patients by diagnosis:
diagnosis
Malignant    57
Benign       24
Name: count, dtype: int64

40X images by subtype:
subtype
Ductal carcinoma       864
Fibroadenoma           253
Mucinous carcinoma     205
Lobular carcinoma      156
Tubular adenoma        149
Papillary carcinoma    145
Adenosis               114
Phyllodes tumor        109
Name: count, dtype: int64

40X patients by subtype:
subtype
Ductal carcinoma       38
Fibroadenoma           10
Mucinous carcinoma      9
Tubular adenoma         7
Papillary carcinoma     6
Lobular carcinoma       5
Adenosis                4
Phyllodes tumor         3
Name: count, dtype: int64


In [ ]:
# Find patient IDs that appear under more than one diagnosis or subtype

patient_check = (
    breakhis_40x
    .groupby("patient_id")
    .agg(
        n_diagnoses=("diagnosis", "nunique"),
        n_subtypes=("subtype", "nunique"),
        diagnoses=("diagnosis", lambda x: sorted(set(x))),
        subtypes=("subtype", lambda x: sorted(set(x))),
        n_images=("filename", "count")
    )
    .reset_index()
)

problem_ids = patient_check[
    (patient_check["n_diagnoses"] > 1) |
    (patient_check["n_subtypes"] > 1)
]

problem_ids

,patient_id,n_diagnoses,n_subtypes,diagnoses,subtypes,n_images
10,14-13412,1,2,[Malignant],"[Ductal carcinoma, Lobular carcinoma]",64


In [ ]:
for pid in problem_ids["patient_id"]:

    print("\nPATIENT ID:", pid)

    display(
        breakhis_40x[
            breakhis_40x["patient_id"] == pid
        ][
            [
                "filename",
                "diagnosis",
                "subtype",
                "magnification"
            ]
        ].sort_values("filename")
    )


PATIENT ID: 14-13412


,filename,diagnosis,subtype,magnification
1185,SOB_M_DC-14-13412-40-001.png,Malignant,Ductal carcinoma,40
1177,SOB_M_DC-14-13412-40-002.png,Malignant,Ductal carcinoma,40
1191,SOB_M_DC-14-13412-40-003.png,Malignant,Ductal carcinoma,40
1193,SOB_M_DC-14-13412-40-004.png,Malignant,Ductal carcinoma,40
1188,SOB_M_DC-14-13412-40-005.png,Malignant,Ductal carcinoma,40
...,...,...,...,...
1705,SOB_M_LC-14-13412-40-028.png,Malignant,Lobular carcinoma,40
1696,SOB_M_LC-14-13412-40-029.png,Malignant,Lobular carcinoma,40
1712,SOB_M_LC-14-13412-40-030.png,Malignant,Lobular carcinoma,40
1695,SOB_M_LC-14-13412-40-031.png,Malignant,Lobular carcinoma,40


In [ ]:
folds_path = root / "Folds.csv"

folds = pd.read_csv(folds_path)

print("Shape:", folds.shape)
print("\nColumns:")
print(folds.columns.tolist())

folds.head(10)

Shape: (39545, 4)

Columns:
['fold', 'mag', 'grp', 'filename']


,fold,mag,grp,filename
0,1,100,train,BreaKHis_v1/histology_slides/breast/benign/SOB...
1,1,100,train,BreaKHis_v1/histology_slides/breast/benign/SOB...
2,1,100,train,BreaKHis_v1/histology_slides/breast/benign/SOB...
3,1,100,train,BreaKHis_v1/histology_slides/breast/benign/SOB...
4,1,100,train,BreaKHis_v1/histology_slides/breast/benign/SOB...
5,1,100,train,BreaKHis_v1/histology_slides/breast/benign/SOB...
6,1,100,train,BreaKHis_v1/histology_slides/breast/benign/SOB...
7,1,100,train,BreaKHis_v1/histology_slides/breast/benign/SOB...
8,1,100,train,BreaKHis_v1/histology_slides/breast/benign/SOB...
9,1,100,train,BreaKHis_v1/histology_slides/breast/benign/SOB...


In [ ]:
print(folds.dtypes)
print()

for col in folds.columns:
    print(f"\n--- {col} ---")
    print(folds[col].head())

fold         int64
mag          int64
grp         object
filename    object
dtype: object


--- fold ---
0    1
1    1
2    1
3    1
4    1
Name: fold, dtype: int64

--- mag ---
0    100
1    100
2    100
3    100
4    100
Name: mag, dtype: int64

--- grp ---
0    train
1    train
2    train
3    train
4    train
Name: grp, dtype: object

--- filename ---
0    BreaKHis_v1/histology_slides/breast/benign/SOB...
1    BreaKHis_v1/histology_slides/breast/benign/SOB...
2    BreaKHis_v1/histology_slides/breast/benign/SOB...
3    BreaKHis_v1/histology_slides/breast/benign/SOB...
4    BreaKHis_v1/histology_slides/breast/benign/SOB...
Name: filename, dtype: object


In [ ]:
import hashlib
import numpy as np
from PIL import Image

def pixel_hash(filepath):
    """
    Hash RGB pixel values rather than the PNG file bytes.
    This detects identical images even if PNG metadata differs.
    """
    with Image.open(filepath) as img:
        arr = np.array(img.convert("RGB"))

    return hashlib.sha256(arr.tobytes()).hexdigest()

In [ ]:
patient_13412 = metadata[
    metadata["patient_id"] == "14-13412"
].copy()

patient_13412["pixel_hash"] = patient_13412["filepath"].apply(pixel_hash)

print("Total files for 14-13412:", len(patient_13412))

print("\nFiles by subtype:")
print(
    patient_13412["subtype"]
    .value_counts()
)

print("\nFiles by magnification and subtype:")
print(
    patient_13412
    .groupby(["magnification", "subtype"])
    .size()
)

Total files for 14-13412: 246

Files by subtype:
subtype
Ductal carcinoma     123
Lobular carcinoma    123
Name: count, dtype: int64

Files by magnification and subtype:
magnification  subtype          
40             Ductal carcinoma     32
               Lobular carcinoma    32
100            Ductal carcinoma     33
               Lobular carcinoma    33
200            Ductal carcinoma     32
               Lobular carcinoma    32
400            Ductal carcinoma     26
               Lobular carcinoma    26
dtype: int64


In [ ]:
comparison = (
    patient_13412
    .groupby(["magnification", "image_number"])
    .agg(
        n_files=("filename", "size"),
        n_subtypes=("subtype", "nunique"),
        n_unique_images=("pixel_hash", "nunique"),
        subtypes=("subtype", lambda x: sorted(set(x))),
        filenames=("filename", list)
    )
    .reset_index()
)

paired = comparison[
    comparison["n_subtypes"] == 2
]

print("DC/LC paired positions:", len(paired))

print(
    "\nPixel-identical DC/LC pairs:",
    (paired["n_unique_images"] == 1).sum()
)

print(
    "Non-identical DC/LC pairs:",
    (paired["n_unique_images"] > 1).sum()
)

DC/LC paired positions: 121

Pixel-identical DC/LC pairs: 89
Non-identical DC/LC pairs: 32


In [ ]:
paired[
    paired["n_unique_images"] > 1
]

,magnification,image_number,n_files,n_subtypes,n_unique_images,subtypes,filenames
76,200,12,2,2,2,"[Ductal carcinoma, Lobular carcinoma]","[SOB_M_DC-14-13412-200-012.png, SOB_M_LC-14-13..."
77,200,13,2,2,2,"[Ductal carcinoma, Lobular carcinoma]","[SOB_M_DC-14-13412-200-013.png, SOB_M_LC-14-13..."
78,200,14,2,2,2,"[Ductal carcinoma, Lobular carcinoma]","[SOB_M_DC-14-13412-200-014.png, SOB_M_LC-14-13..."
79,200,15,2,2,2,"[Ductal carcinoma, Lobular carcinoma]","[SOB_M_DC-14-13412-200-015.png, SOB_M_LC-14-13..."
80,200,16,2,2,2,"[Ductal carcinoma, Lobular carcinoma]","[SOB_M_DC-14-13412-200-016.png, SOB_M_LC-14-13..."
81,200,17,2,2,2,"[Ductal carcinoma, Lobular carcinoma]","[SOB_M_DC-14-13412-200-017.png, SOB_M_LC-14-13..."
82,200,18,2,2,2,"[Ductal carcinoma, Lobular carcinoma]","[SOB_M_DC-14-13412-200-018.png, SOB_M_LC-14-13..."
83,200,19,2,2,2,"[Ductal carcinoma, Lobular carcinoma]","[SOB_M_DC-14-13412-200-019.png, SOB_M_LC-14-13..."
84,200,20,2,2,2,"[Ductal carcinoma, Lobular carcinoma]","[SOB_M_DC-14-13412-200-020.png, SOB_M_LC-14-13..."
85,200,21,2,2,2,"[Ductal carcinoma, Lobular carcinoma]","[SOB_M_DC-14-13412-200-021.png, SOB_M_LC-14-13..."


In [ ]:
patient_13412_40x = patient_13412[
    patient_13412["magnification"] == 40
].copy()

comparison_40x = (
    patient_13412_40x
    .groupby("image_number")
    .agg(
        n_files=("filename", "size"),
        n_subtypes=("subtype", "nunique"),
        n_unique_images=("pixel_hash", "nunique"),
        subtypes=("subtype", lambda x: sorted(set(x))),
        filenames=("filename", list)
    )
    .reset_index()
)

print("Total 40X files:", len(patient_13412_40x))
print("40X paired positions:", len(comparison_40x))

print(
    "Pixel-identical 40X DC/LC pairs:",
    (comparison_40x["n_unique_images"] == 1).sum()
)

print(
    "Non-identical 40X DC/LC pairs:",
    (comparison_40x["n_unique_images"] > 1).sum()
)

comparison_40x[
    comparison_40x["n_unique_images"] > 1
]

Total 40X files: 64
40X paired positions: 32
Pixel-identical 40X DC/LC pairs: 32
Non-identical 40X DC/LC pairs: 0


,image_number,n_files,n_subtypes,n_unique_images,subtypes,filenames


In [ ]:
# Add pixel hashes to the complete 40X dataset
breakhis_40x = breakhis_40x.copy()

breakhis_40x["pixel_hash"] = (
    breakhis_40x["filepath"]
    .apply(pixel_hash)
)

# Find every pixel hash appearing more than once
duplicate_hash_counts = (
    breakhis_40x["pixel_hash"]
    .value_counts()
)

duplicate_hashes = duplicate_hash_counts[
    duplicate_hash_counts > 1
].index

duplicate_rows = (
    breakhis_40x[
        breakhis_40x["pixel_hash"].isin(duplicate_hashes)
    ]
    .sort_values(["pixel_hash", "filename"])
)

print("Duplicate pixel-hash groups:", len(duplicate_hashes))
print("Total files involved in duplicate groups:", len(duplicate_rows))

print("\nPatient IDs involved:")
print(duplicate_rows["patient_id"].value_counts())

print("\nSubtypes involved:")
print(duplicate_rows["subtype"].value_counts())

Duplicate pixel-hash groups: 34
Total files involved in duplicate groups: 68

Patient IDs involved:
patient_id
14-13412    64
14-12204     4
Name: count, dtype: int64

Subtypes involved:
subtype
Lobular carcinoma    36
Ductal carcinoma     32
Name: count, dtype: int64


In [ ]:
dup_12204 = duplicate_rows[
    duplicate_rows["patient_id"] == "14-12204"
].copy()

display(
    dup_12204[
        [
            "filename",
            "patient_id",
            "diagnosis",
            "subtype",
            "magnification",
            "image_number",
            "pixel_hash"
        ]
    ].sort_values(["pixel_hash", "image_number"])
)

,filename,patient_id,diagnosis,subtype,magnification,image_number,pixel_hash
1723,SOB_M_LC-14-12204-40-017.png,14-12204,Malignant,Lobular carcinoma,40,17,7ff88c3a6496f0bd7bb8aef6f9921763c35a798e35ba66...
1737,SOB_M_LC-14-12204-40-035.png,14-12204,Malignant,Lobular carcinoma,40,35,7ff88c3a6496f0bd7bb8aef6f9921763c35a798e35ba66...
1722,SOB_M_LC-14-12204-40-016.png,14-12204,Malignant,Lobular carcinoma,40,16,c5bea6c04c60d094c75a31029a48734712260e3cef0325...
1734,SOB_M_LC-14-12204-40-034.png,14-12204,Malignant,Lobular carcinoma,40,34,c5bea6c04c60d094c75a31029a48734712260e3cef0325...


In [ ]:
pairs_12204 = (
    dup_12204
    .groupby("pixel_hash")
    .agg(
        n_files=("filename", "size"),
        filenames=("filename", list),
        image_numbers=("image_number", list),
        magnifications=("magnification", list),
        subtypes=("subtype", lambda x: sorted(set(x)))
    )
    .reset_index()
)

pairs_12204

,pixel_hash,n_files,filenames,image_numbers,magnifications,subtypes
0,7ff88c3a6496f0bd7bb8aef6f9921763c35a798e35ba66...,2,"[SOB_M_LC-14-12204-40-017.png, SOB_M_LC-14-122...","[17, 35]","[40, 40]",[Lobular carcinoma]
1,c5bea6c04c60d094c75a31029a48734712260e3cef0325...,2,"[SOB_M_LC-14-12204-40-016.png, SOB_M_LC-14-122...","[16, 34]","[40, 40]",[Lobular carcinoma]


In [ ]:
import hashlib

def file_sha256(filepath):
    sha = hashlib.sha256()

    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            sha.update(chunk)

    return sha.hexdigest()

dup_12204["file_hash"] = (
    dup_12204["filepath"]
    .apply(file_sha256)
)

display(
    dup_12204[
        [
            "filename",
            "image_number",
            "pixel_hash",
            "file_hash"
        ]
    ].sort_values(["pixel_hash", "image_number"])
)

,filename,image_number,pixel_hash,file_hash
1723,SOB_M_LC-14-12204-40-017.png,17,7ff88c3a6496f0bd7bb8aef6f9921763c35a798e35ba66...,85b2c1b6ad2e36a34880bdadad5074b7a6a1af57df9e29...
1737,SOB_M_LC-14-12204-40-035.png,35,7ff88c3a6496f0bd7bb8aef6f9921763c35a798e35ba66...,2d624f689074250fe3456f7db2f7e46a0a8860548e23df...
1722,SOB_M_LC-14-12204-40-016.png,16,c5bea6c04c60d094c75a31029a48734712260e3cef0325...,9c8da669281ab3a5d374fc9ae8fe1b73e411d15c87c379...
1734,SOB_M_LC-14-12204-40-034.png,34,c5bea6c04c60d094c75a31029a48734712260e3cef0325...,2cf5f347ac84edeb97c78548b9080c6512ef0460574142...


In [ ]:
print(
    dup_12204
    .groupby("pixel_hash")["file_hash"]
    .nunique()
)

pixel_hash
7ff88c3a6496f0bd7bb8aef6f9921763c35a798e35ba6652530b153c3264169f    2
c5bea6c04c60d094c75a31029a48734712260e3cef03258140e1fb5912f7b564    2
Name: file_hash, dtype: int64


## 3. BreakHis 40X Data Quality Audit

Exact pixel-level duplicate detection was performed on the 1,995 BreakHis
40X images using SHA-256 hashes of the decoded RGB pixel arrays.

A total of 34 duplicate-image groups involving 68 files were identified.

Two patient identifiers contained all detected duplicates:

- `14-13412`: 32 duplicate pairs involving ductal carcinoma and lobular
  carcinoma filenames. For the 40X subset, all 32 corresponding DC/LC pairs
  contained identical decoded RGB pixel values.
- `14-12204`: 2 duplicate pairs within the lobular carcinoma class
  (image 16 vs. 34 and image 17 vs. 35). The PNG files were not byte-identical,
  but their decoded RGB pixel values were identical.

To prevent duplicate weighting and possible leakage, only one image from each
within-patient pixel-identical group is retained for analysis.

Patient `14-13412` is treated as one malignant patient group and its subtype is
recorded as `DC/LC ambiguous` because the same 40X images occur under both
ductal and lobular carcinoma filenames.

Raw image files are not modified. All corrections are performed only in the
analysis metadata.

In [ ]:
# Start again from the untouched 40X metadata
analysis_40x = breakhis_40x.copy()

# Preserve the dataset's original subtype labels
analysis_40x["original_subtype_code"] = analysis_40x["subtype_code"]
analysis_40x["original_subtype"] = analysis_40x["subtype"]

# Deterministic ordering
analysis_40x = (
    analysis_40x
    .sort_values(
        ["patient_id", "pixel_hash", "filename"]
    )
    .reset_index(drop=True)
)

# Flag pixel-identical duplicates within the same patient
analysis_40x["is_pixel_duplicate"] = (
    analysis_40x
    .duplicated(
        subset=["patient_id", "pixel_hash"],
        keep="first"
    )
)

print("Duplicate rows:", analysis_40x["is_pixel_duplicate"].sum())

Duplicate rows: 34


In [ ]:
mask_13412 = analysis_40x["patient_id"] == "14-13412"

analysis_40x.loc[
    mask_13412,
    "subtype"
] = "DC/LC ambiguous"

analysis_40x.loc[
    mask_13412,
    "subtype_code"
] = "DC/LC"

In [ ]:
duplicate_audit = analysis_40x[
    analysis_40x["is_pixel_duplicate"]
].copy()

clean_40x = (
    analysis_40x[
        ~analysis_40x["is_pixel_duplicate"]
    ]
    .copy()
    .reset_index(drop=True)
)

In [ ]:
clean_40x["relative_path"] = clean_40x["filepath"].apply(
    lambda x: str(Path(x).relative_to(root))
)

metadata_to_save = clean_40x[
    [
        "filename",
        "relative_path",
        "patient_id",
        "diagnosis_code",
        "diagnosis",
        "original_subtype_code",
        "original_subtype",
        "subtype_code",
        "subtype",
        "magnification",
        "image_number"
    ]
].copy()

metadata_to_save.to_csv(
    metadata_dir / "breakhis_40x_clean_metadata.csv",
    index=False
)

In [ ]:
duplicate_audit["relative_path"] = duplicate_audit["filepath"].apply(
    lambda x: str(Path(x).relative_to(root))
)

duplicate_audit_to_save = duplicate_audit[
    [
        "filename",
        "relative_path",
        "patient_id",
        "diagnosis",
        "original_subtype",
        "subtype",
        "magnification",
        "image_number",
        "pixel_hash"
    ]
].copy()

duplicate_audit_to_save["exclusion_reason"] = (
    "Within-patient pixel-identical duplicate"
)

duplicate_audit_to_save.to_csv(
    metadata_dir / "breakhis_40x_duplicate_audit.csv",
    index=False
)

In [ ]:
print("=== FINAL CLEAN BREAKHIS 40X ===")

print("\nImages:")
print(len(clean_40x))

print("\nUnique patient groups:")
print(clean_40x["patient_id"].nunique())

print("\nImages by diagnosis:")
print(clean_40x["diagnosis"].value_counts())

patient_level = (
    clean_40x[
        ["patient_id", "diagnosis"]
    ]
    .drop_duplicates()
)

print("\nPatients by diagnosis:")
print(patient_level["diagnosis"].value_counts())

print("\nDuplicate flag remaining:")
print(
    clean_40x
    .duplicated(
        subset=["patient_id", "pixel_hash"]
    )
    .sum()
)

=== FINAL CLEAN BREAKHIS 40X ===

Images:
1961

Unique patient groups:
81

Images by diagnosis:
diagnosis
Malignant    1336
Benign        625
Name: count, dtype: int64

Patients by diagnosis:
diagnosis
Malignant    57
Benign       24
Name: count, dtype: int64

Duplicate flag remaining:
0


In [ ]:
from pathlib import Path

# Confirm the updated cleaning pipeline preserved the original subtype fields
required_columns = [
    "original_subtype_code",
    "original_subtype"
]

for col in required_columns:
    assert col in clean_40x.columns, f"Missing required column: {col}"

# Store paths relative to the BreakHis dataset root
clean_40x["relative_path"] = clean_40x["filepath"].apply(
    lambda x: str(Path(x).relative_to(root))
)

duplicate_audit["relative_path"] = duplicate_audit["filepath"].apply(
    lambda x: str(Path(x).relative_to(root))
)

# Final clean metadata table
metadata_to_save = clean_40x[
    [
        "filename",
        "relative_path",
        "patient_id",
        "diagnosis_code",
        "diagnosis",
        "original_subtype_code",
        "original_subtype",
        "subtype_code",
        "subtype",
        "magnification",
        "image_number"
    ]
].copy()

# Duplicate audit table
duplicate_audit_to_save = duplicate_audit[
    [
        "filename",
        "relative_path",
        "patient_id",
        "diagnosis",
        "original_subtype",
        "subtype",
        "magnification",
        "image_number",
        "pixel_hash"
    ]
].copy()

duplicate_audit_to_save["exclusion_reason"] = (
    "Within-patient pixel-identical duplicate"
)

print("Clean metadata shape:", metadata_to_save.shape)
print("Duplicate audit shape:", duplicate_audit_to_save.shape)

print("\nClean metadata columns:")
print(metadata_to_save.columns.tolist())

Clean metadata shape: (1961, 11)
Duplicate audit shape: (34, 10)

Clean metadata columns:
['filename', 'relative_path', 'patient_id', 'diagnosis_code', 'diagnosis', 'original_subtype_code', 'original_subtype', 'subtype_code', 'subtype', 'magnification', 'image_number']


In [ ]:
from pathlib import Path

metadata_dir = Path(
    "/content/breast-histology-calibration/metadata"
)

metadata_dir.mkdir(
    parents=True,
    exist_ok=True
)

clean_metadata_path = (
    metadata_dir / "breakhis_40x_clean_metadata.csv"
)

duplicate_audit_path = (
    metadata_dir / "breakhis_40x_duplicate_audit.csv"
)

metadata_to_save.to_csv(
    clean_metadata_path,
    index=False
)

duplicate_audit_to_save.to_csv(
    duplicate_audit_path,
    index=False
)

print("Saved clean metadata:")
print(clean_metadata_path)

print("\nSaved duplicate audit:")
print(duplicate_audit_path)

Saved clean metadata:
/content/breast-histology-calibration/metadata/breakhis_40x_clean_metadata.csv

Saved duplicate audit:
/content/breast-histology-calibration/metadata/breakhis_40x_duplicate_audit.csv


In [ ]:
# Reload the saved files to verify they were written correctly

saved_clean = pd.read_csv(clean_metadata_path)
saved_audit = pd.read_csv(duplicate_audit_path)

print("Saved clean metadata shape:", saved_clean.shape)
print("Saved duplicate audit shape:", saved_audit.shape)

print("\nUnique patients in saved clean metadata:")
print(saved_clean["patient_id"].nunique())

print("\nDiagnosis counts:")
print(saved_clean["diagnosis"].value_counts())

print("\nMissing values by column:")
print(saved_clean.isna().sum())

Saved clean metadata shape: (1961, 11)
Saved duplicate audit shape: (34, 10)

Unique patients in saved clean metadata:
81

Diagnosis counts:
diagnosis
Malignant    1336
Benign        625
Name: count, dtype: int64

Missing values by column:
filename                 0
relative_path            0
patient_id               0
diagnosis_code           0
diagnosis                0
original_subtype_code    0
original_subtype         0
subtype_code             0
subtype                  0
magnification            0
image_number             0
dtype: int64


In [ ]:
# ============================================================
# 4. Build patient-level metadata for splitting
# ============================================================

patient_table = (
    saved_clean[
        [
            "patient_id",
            "diagnosis_code",
            "diagnosis",
            "subtype_code",
            "subtype"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["diagnosis", "subtype", "patient_id"]
    )
    .reset_index(drop=True)
)

print("Total patient groups:")
print(len(patient_table))

print("\nPatients by diagnosis:")
print(
    patient_table["diagnosis"]
    .value_counts()
)

print("\nPatients by subtype:")
print(
    patient_table["subtype"]
    .value_counts()
)

patient_table.head(10)

Total patient groups:
81

Patients by diagnosis:
diagnosis
Malignant    57
Benign       24
Name: count, dtype: int64

Patients by subtype:
subtype
Ductal carcinoma       37
Fibroadenoma           10
Mucinous carcinoma      9
Tubular adenoma         7
Papillary carcinoma     6
Adenosis                4
Lobular carcinoma       4
Phyllodes tumor         3
DC/LC ambiguous         1
Name: count, dtype: int64


,patient_id,diagnosis_code,diagnosis,subtype_code,subtype
0,14-22549AB,B,Benign,A,Adenosis
1,14-22549CD,B,Benign,A,Adenosis
2,14-22549G,B,Benign,A,Adenosis
3,14-29960CD,B,Benign,A,Adenosis
4,14-14134,B,Benign,F,Fibroadenoma
5,14-14134E,B,Benign,F,Fibroadenoma
6,14-21998CD,B,Benign,F,Fibroadenoma
7,14-21998EF,B,Benign,F,Fibroadenoma
8,14-23060AB,B,Benign,F,Fibroadenoma
9,14-23060CD,B,Benign,F,Fibroadenoma


In [ ]:
# ============================================================
# 5. Create fixed patient-level Development / Calibration / Test split
# ============================================================

import numpy as np
import pandas as pd

SPLIT_SEED = 631
rng = np.random.default_rng(SPLIT_SEED)

# Exact subtype allocation based on the verified 81-patient dataset
split_targets = {
    "Adenosis": {
        "development": 2,
        "calibration": 1,
        "test": 1
    },
    "Fibroadenoma": {
        "development": 6,
        "calibration": 2,
        "test": 2
    },
    "Tubular adenoma": {
        "development": 5,
        "calibration": 1,
        "test": 1
    },
    "Phyllodes tumor": {
        "development": 1,
        "calibration": 1,
        "test": 1
    },
    "Ductal carcinoma": {
        "development": 23,
        "calibration": 7,
        "test": 7
    },
    "Lobular carcinoma": {
        "development": 2,
        "calibration": 1,
        "test": 1
    },
    "Mucinous carcinoma": {
        "development": 5,
        "calibration": 2,
        "test": 2
    },
    "Papillary carcinoma": {
        "development": 4,
        "calibration": 1,
        "test": 1
    },
    "DC/LC ambiguous": {
        "development": 1,
        "calibration": 0,
        "test": 0
    }
}

split_records = []

for subtype, targets in split_targets.items():

    subtype_patients = (
        patient_table[
            patient_table["subtype"] == subtype
        ]
        .copy()
        .sort_values("patient_id")
        .reset_index(drop=True)
    )

    expected_total = sum(targets.values())

    assert len(subtype_patients) == expected_total, (
        f"{subtype}: expected {expected_total} patients, "
        f"found {len(subtype_patients)}"
    )

    # Fixed random permutation within subtype
    shuffled_indices = rng.permutation(len(subtype_patients))

    subtype_patients = (
        subtype_patients
        .iloc[shuffled_indices]
        .reset_index(drop=True)
    )

    start = 0

    for split_name in ["development", "calibration", "test"]:

        n = targets[split_name]

        selected = subtype_patients.iloc[start:start+n].copy()
        selected["split"] = split_name

        split_records.append(selected)

        start += n

patient_split = (
    pd.concat(split_records, ignore_index=True)
    .sort_values(["split", "diagnosis", "subtype", "patient_id"])
    .reset_index(drop=True)
)

print("Total assigned patients:", len(patient_split))
patient_split.head()

Total assigned patients: 81


,patient_id,diagnosis_code,diagnosis,subtype_code,subtype,split
0,14-22549G,B,Benign,A,Adenosis,calibration
1,14-23060CD,B,Benign,F,Fibroadenoma,calibration
2,14-29960AB,B,Benign,F,Fibroadenoma,calibration
3,14-22704,B,Benign,PT,Phyllodes tumor,calibration
4,14-21978AB,B,Benign,TA,Tubular adenoma,calibration


In [ ]:
# ============================================================
# Verify patient split
# ============================================================

print("=== PATIENT SPLIT SUMMARY ===")

print("\nPatients per split:")
print(
    patient_split["split"]
    .value_counts()
)

print("\nDiagnosis by split:")
print(
    pd.crosstab(
        patient_split["split"],
        patient_split["diagnosis"]
    )
)

print("\nSubtype by split:")
print(
    pd.crosstab(
        patient_split["subtype"],
        patient_split["split"]
    )
)

# Make sure every patient appears exactly once
assert patient_split["patient_id"].nunique() == 81
assert len(patient_split) == 81
assert patient_split["patient_id"].duplicated().sum() == 0

# Exact split-size checks
assert (patient_split["split"] == "development").sum() == 49
assert (patient_split["split"] == "calibration").sum() == 16
assert (patient_split["split"] == "test").sum() == 16

print("\nPASS: All 81 patients assigned exactly once.")
print("PASS: Development = 49 patients.")
print("PASS: Calibration = 16 patients.")
print("PASS: Test = 16 patients.")

=== PATIENT SPLIT SUMMARY ===

Patients per split:
split
development    49
calibration    16
test           16
Name: count, dtype: int64

Diagnosis by split:
diagnosis    Benign  Malignant
split                         
calibration       5         11
development      14         35
test              5         11

Subtype by split:
split                calibration  development  test
subtype                                            
Adenosis                       1            2     1
DC/LC ambiguous                0            1     0
Ductal carcinoma               7           23     7
Fibroadenoma                   2            6     2
Lobular carcinoma              1            2     1
Mucinous carcinoma             2            5     2
Papillary carcinoma            1            4     1
Phyllodes tumor                1            1     1
Tubular adenoma                1            5     1

PASS: All 81 patients assigned exactly once.
PASS: Development = 49 patients.
PASS: Calibration